# 我們平台訓練 — Detecting Reversal Points in US Equities

3-class 時序分類 (H / L / None),極不平衡 (94 / 3 / 3),~2684 樣本 × 68503 特徵。
本 notebook 直接 call 我們平台的 **daniel pipeline**(雙軌 Tabular + DL ensemble + Nelder-Mead blender + Meta-learner stacker),最後輸出 `/kaggle/working/submission.csv`。

## 前置作業 (一次性)
1. 把專案的 `api/` 整個資料夾上傳成一份 Kaggle Dataset(名字隨意,notebook 會自動偵測)
2. 本 notebook 的 **Settings → Add Input** → 加上這份 dataset
3. 同樣 attach 比賽資料 `Detecting Reversal Points in US Equities`
4. (可選但建議) Accelerator 選 GPU T4 x2 或 P100 — daniel 的 DL 軌道會用上

## Step 1 ─ 環境準備
找到平台 code dataset → 設 sys.path → 補上 Kaggle 沒預裝的依賴。

In [ ]:
import sys, os, subprocess

# 找平台 code dataset (要含 api/train/pipeline 才算)
PLATFORM_ROOT = None
for d in os.listdir('/kaggle/input'):
    p = os.path.join('/kaggle/input', d)
    if os.path.isdir(os.path.join(p, 'api', 'train', 'pipeline')):
        PLATFORM_ROOT = p
        break
assert PLATFORM_ROOT, '找不到平台 code — 請先 attach 你上傳的 api/ 那份 Kaggle Dataset'
print(f'[platform] root = {PLATFORM_ROOT}')

# pipeline_time / pipeline.py 在 api/train/pipeline/ 底下當頂層 module 用,要加路徑
sys.path.insert(0, PLATFORM_ROOT)
sys.path.insert(0, os.path.join(PLATFORM_ROOT, 'api', 'train', 'pipeline'))

# Kaggle base image 已預裝大部分依賴 (lightgbm/xgboost/catboost/torch/optuna/sklearn);
# 平台部分 module 會用 imblearn / shap,補一下
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'imbalanced-learn', 'shap'], check=False)
print('[deps] OK')

In [ ]:
# 比賽資料自動偵測:掃 /kaggle/input,找含 train.csv+test.csv 的資料夾,跳過 platform dataset
# 多個候選時優先 'new' 命名(舊比賽常見 new_xxx 為最新版資料)
def find_comp_data():
    candidates = []
    for dirpath, _dirs, files in os.walk('/kaggle/input'):
        if PLATFORM_ROOT in dirpath:
            continue
        if 'train.csv' in files and 'test.csv' in files:
            size = os.path.getsize(os.path.join(dirpath, 'train.csv'))
            candidates.append((dirpath, size))
    if not candidates:
        return None
    candidates.sort(key=lambda x: (0 if 'new' in os.path.basename(x[0]).lower() else 1, -x[1]))
    return candidates[0][0]

DATA_DIR = find_comp_data()
assert DATA_DIR, '找不到比賽資料 — 請 attach Detecting Reversal Points in US Equities'
print(f'[data] {DATA_DIR}')

# GPU 偵測
def has_gpu():
    try:
        r = subprocess.run(['nvidia-smi'], capture_output=True, timeout=3)
        return r.returncode == 0
    except Exception:
        return False
HAS_GPU = has_gpu()
print(f'[gpu]  {HAS_GPU}')

## Step 2 ─ 載入資料
Reversal Points 高維 binary 特徵,float32 壓縮一半記憶體。

In [ ]:
import pandas as pd, numpy as np
import time, warnings, gc
warnings.filterwarnings('ignore')

TARGET = 'class_label'
META   = {'train_id', 'id', 'ticker_id', 't'}

print('讀 train ...')
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'), low_memory=False)
print(f'  train: {train_df.shape}')
print('讀 test ...')
test_df = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'), low_memory=False)
print(f'  test:  {test_df.shape}')

print(f'\n[label] 分佈:\n{train_df[TARGET].value_counts(dropna=False)}')

## Step 3 ─ 特徵準備 + label 編碼
Reversal 的特徵全是純數值 binary (cross_threshold / happens_within...),不需要 OHE。
只要扣掉 metadata + 對齊 train/test 欄即可。
Label 用 LabelEncoder 編成 0/1/2 給 daniel 用,最後預測再 inverse_transform 還原成 H/L/None。

In [ ]:
from sklearn.preprocessing import LabelEncoder

# 同時存在於 train + test 的純數值特徵
feat_cols = [c for c in train_df.columns
             if c not in META and c != TARGET and c in test_df.columns
             and pd.api.types.is_numeric_dtype(train_df[c])]
print(f'[feat] {len(feat_cols)} 個數值特徵')

X_tr = train_df[feat_cols].fillna(0).values.astype(np.float32)
X_te = test_df[feat_cols].fillna(0).values.astype(np.float32)
test_ids = test_df['id'].values

# 5-class → 3-class 映射(若 Kaggle 給的還是原 5 類)
FIVE_TO_THREE = {'HH': 'H', 'LH': 'H', 'HL': 'L', 'LL': 'L'}
y_raw = train_df[TARGET].astype(str).replace(FIVE_TO_THREE).replace({'nan': 'None'}).fillna('None')
print(f'[label] 映射後類別: {sorted(y_raw.unique().tolist())}')

le = LabelEncoder()
y_tr = le.fit_transform(y_raw.values).astype(np.int64)
CLASSES = le.classes_
NUM_CLASSES = len(CLASSES)
print(f'[label] classes = {dict(enumerate(CLASSES))}')
print(f'[shape] X_tr={X_tr.shape}  X_te={X_te.shape}  y_tr={y_tr.shape}')

del train_df, test_df
gc.collect()

## Step 4 ─ 跑 daniel pipeline
雙軌 Tabular + DL ensemble,內部跑 Optuna HPO + StratifiedKFold + Nelder-Mead blender + Meta-learner stacker。

Reversal 是 **2684 列 × 68k 特徵**(P >> N),DL 軌容易過擬合,先 `skip_dl=True` 只跑 tabular。要連 DL 一起跑就改 False(會慢很多)。

In [ ]:
import pipeline_time as pt
import pipeline as pl

cfg = pt.get_cfg_time(fast=False, n_samples=len(y_tr))
budget = pl.TimeBudget(limit_sec=4500, t_start=time.time())   # 75 分鐘上限

print(f'[cfg] tabular_trials={cfg.get("tabular_trials")} dl_trials={cfg.get("dl_trials")}')
print(f'[budget] 75 min')

# run_classification 內部 delegate 給 pipeline.run(..., is_ts=True),簽名:
#   (X_train, y_train, X_test, n_classes, cfg, budget, *, skip_tabular, skip_dl, no_nas, artifacts_dir, metric)
result = pt.run_classification(
    X_tr, y_tr, X_te,
    NUM_CLASSES,
    cfg, budget,
    skip_tabular=False,
    skip_dl=True,                              # 高維小樣本 DL 易過擬合,demo 先跳過
    no_nas=True,                                # 同理 NAS 沒幫助
    artifacts_dir='/kaggle/working/artifacts',
    metric='f1',
)
print(f'\n[done] {len(result.model_tags)} 個 base model 訓練完')
print(f'[models] {result.model_tags}')

## Step 5 ─ OOF 評估 + 每個 base model 排行榜

In [ ]:
from sklearn.metrics import f1_score, classification_report
import matplotlib.pyplot as plt

# Per-model OOF macro-F1
per_model = []
for tag, oof_pred in zip(result.model_tags, result.all_oof):
    oa = np.asarray(oof_pred)
    if oa.ndim == 2:
        idx = np.argmax(oa, axis=1)
    else:
        idx = oa.astype(int)
    try:
        s = f1_score(y_tr, idx, average='macro')
        per_model.append((tag, float(s)))
    except Exception:
        pass
per_model.sort(key=lambda x: x[1], reverse=True)

print('[Per-Model OOF Leaderboard]')
for tag, s in per_model:
    print(f'  {s:.4f}  {tag}')

# 簡單把 all_oof 平均當 OOF ensemble proxy(stacker.predict_oof 是 fit 用的,不能直接拿來評估)
oof_avg = np.mean([np.asarray(o) for o in result.all_oof], axis=0)
oof_pred = np.argmax(oof_avg, axis=1)
print(f'\n[Ensemble (avg) OOF macro-F1] {f1_score(y_tr, oof_pred, average="macro"):.4f}')
print(f'\n[classification_report]\n{classification_report(y_tr, oof_pred, target_names=CLASSES, zero_division=0)}')

# Bar chart
tags = [t for t, _ in per_model]
scores = [s for _, s in per_model]
plt.figure(figsize=(10, max(3, len(tags) * 0.35)))
plt.barh(tags, scores, color='steelblue')
plt.xlabel('OOF macro-F1')
plt.title('Daniel Pipeline — Per-Model OOF Leaderboard')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## Step 6 ─ 圖表:Confusion matrix + 各類預測分布

In [ ]:
from sklearn.metrics import confusion_matrix

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# (1) Confusion matrix
cm = confusion_matrix(y_tr, oof_pred)
im = axes[0].imshow(cm, cmap='Blues')
axes[0].set_xticks(range(NUM_CLASSES)); axes[0].set_yticks(range(NUM_CLASSES))
axes[0].set_xticklabels(CLASSES); axes[0].set_yticklabels(CLASSES)
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
axes[0].set_title('Confusion Matrix (OOF ensemble avg)')
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        axes[0].text(j, i, str(cm[i, j]), ha='center', va='center',
                     color='white' if cm[i, j] > cm.max()/2 else 'black')
plt.colorbar(im, ax=axes[0])

# (2) 預測類別分布(test set)
test_probs = np.asarray(result.test_stack)
test_pred_idx = np.argmax(test_probs, axis=1)
pred_counts = pd.Series(test_pred_idx).value_counts().sort_index()
axes[1].bar([CLASSES[i] for i in pred_counts.index], pred_counts.values,
            color=['#3b82f6', '#ef4444', '#10b981'][:NUM_CLASSES])
axes[1].set_ylabel('Count')
axes[1].set_title('Test Set — Predicted Class Distribution (stack)')
for i, v in enumerate(pred_counts.values):
    axes[1].text(i, v, str(v), ha='center', va='bottom')

plt.tight_layout()
plt.show()

## Step 7 ─ 生成 submission.csv
用 daniel 的 **stacker** 作為最終預測(Meta-learner 通常比 Nelder-Mead blender 強)。
若要改用 blender,改 `test_probs = result.test_blend`。

In [ ]:
test_pred_labels = le.inverse_transform(test_pred_idx)

submission = pd.DataFrame({'id': test_ids, 'class_label': test_pred_labels})
out_path = '/kaggle/working/submission.csv'
submission.to_csv(out_path, index=False)
print(f'[saved] {out_path}  shape={submission.shape}')
print(f'\n[head]\n{submission.head()}')
print(f'\n[類別分布]\n{submission["class_label"].value_counts()}')

## Summary

**架構**:Daniel pipeline (tabular ensemble) + StratifiedKFold + Optuna HPO + Nelder-Mead blender + Meta-learner stacker

**輸出**:`/kaggle/working/submission.csv` — 點右上 **Submit to Competition** 即可

**想跑更強**:
- 把 `skip_dl=True` 改 `False` 加上 DL 軌(TCN / PatchTST)
- 把 `fast=False` 維持,或 `cfg['tabular_trials']` 拉大(預設已 10+)
- `limit_sec=4500` 拉到 9000 (2.5h),Kaggle GPU 上限 12h 內都 OK

**Reversal Points 特性提醒**:極不平衡(94/3/3),只看 accuracy 沒意義,**重點是 H 跟 L 的 recall**。